# A.2 — Dự đoán ký tự kế tiếp bằng decoder A.1
**Hai việc riêng:** các ô đầu chạy CPU với chuỗi minh họa tự viết. Ô cuối huấn luyện trên *Truyện Kiều*, bản Quốc ngữ Trương Vĩnh Ký 1911; mặc định tắt cho tới khi nhóm xác nhận đúng manifest dữ liệu. Không dùng QA phần D.

Đặt notebook trong bản repo có `src/`, `scripts/` và `tests/`. Trên Colab, đưa **bản repo của nhóm** vào runtime rồi đặt thư mục làm việc ở gốc repo; không cần cài lại PyTorch. Không tự cấp quyền Drive.

### Từ mô hình xác suất đến loss
Với cửa sổ $c_0,\ldots,c_T$, quy tắc chuỗi cho $p_\theta(c_{1:T}\mid c_0)=\prod_{t=0}^{T-1} p_\theta(c_{t+1}\mid c_{0:t})$. Embedding + vị trí sinusoidal + decoder nhân quả A.1 + phép chiếu sinh logits $z_t$; softmax biến chúng thành phân phối trên từ vựng. Tối đa hóa log-likelihood tương đương tối thiểu hóa $\mathrm{NLL}=\frac1N\sum_t[\log\sum_j e^{z_{t,j}}-z_{t,c_{t+1}}]$, với $N$ là số target thực được chấm. Validation PPL $=\exp(\mathrm{NLL}_{val})$: log tự nhiên, tính trên toàn bộ target của các cửa sổ validation, kể cả UNK; **không** trung bình PPL của từng minibatch. [Đối chiếu: PyTorch cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html).

Bộ chuẩn bị chia văn bản NFC thành hai đoạn **trước** khi tạo cửa sổ. Từ vựng chỉ học từ train; phần đuôi không đủ một cửa sổ không được chấm. Một ký tự là một Unicode code point sau NFC, không phải byte hoặc từ.

In [ ]:
from pathlib import Path
import hashlib, json, platform, subprocess, sys, tempfile
import torch
from torch.nn import functional as F

here = Path.cwd().resolve()
root = next((p for p in (here, *here.parents) if (p / 'scripts/train_char_lm.py').is_file()), None)
if root is None:
    raise RuntimeError('Hãy đặt thư mục làm việc ở gốc bản repo của nhóm trước khi chạy.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.char_lm import CharacterLM
from src.char_lm_data import windows_from_ids
from scripts.train_char_lm import run_experiment
print('Python', platform.python_version(), '| torch', torch.__version__, '| CUDA khả dụng:', torch.cuda.is_available())

### Kiểm lệch một ký tự và gradient trên CPU
`abcd` là dữ liệu minh họa: `abc` phải dự đoán `bcd`, không dự đoán lại `abc`. Đối chiếu NLL dẫn xuất với cross-entropy trên chính logits của mô hình; kiểm gradient tới đầu chiếu Q/K/V bên trong decoder.

In [ ]:
vocab = {'<UNK>': 0, **{ch: i for i, ch in enumerate('abcd', start=1)}}
def ids(text):
    return torch.tensor([vocab.get(ch, 0) for ch in text], dtype=torch.long)

window = windows_from_ids(ids('abcd'), 3)
x, y = window[:, :-1], window[:, 1:]
assert x.tolist() == [[1, 2, 3]] and y.tolist() == [[2, 3, 4]]
with torch.random.fork_rng(devices=[]):
    torch.random.default_generator.manual_seed(41)
    lm = CharacterLM(len(vocab), 3, 8, 2, 16, 1)
    z = lm(x)
    nll = (z.logsumexp(dim=-1) - z.gather(-1, y.unsqueeze(-1)).squeeze(-1)).mean()
    ce = F.cross_entropy(z.flatten(0, 1), y.flatten())
    torch.testing.assert_close(nll, ce)
    ce.backward()
    assert lm.blocks[0].attn.qkv.weight.grad.norm().item() > 0
print('abc → bcd | NLL tính theo logits = cross-entropy:', round(ce.item(), 6))

### Chạy thử đúng vòng huấn luyện và tổng hợp kết quả
Hai đoạn dưới **tự tạo để kiểm mã**, không phải ngữ liệu của đề. `đ` không có trong từ vựng train nên trở thành UNK ở validation. Mỗi seed khởi tạo model/optimizer riêng. Các checkpoint minh họa nằm trong thư mục tạm và sẽ được xóa.

In [ ]:
toy_train, toy_val = 'abcd' * 8, 'bcdđabcdab'
train = windows_from_ids(ids(toy_train), 3)
validation = windows_from_ids(ids(toy_val), 3)
config = dict(d_model=8, n_heads=2, d_ff=16, n_layers=1, epochs=2, batch_size=4, lr=0.01)
data_id = hashlib.sha256(('MINH HOA\n' + toy_train + '\n' + toy_val).encode()).hexdigest()
with tempfile.TemporaryDirectory() as tmp:
    output = Path(tmp) / 'minh-hoa'
    run_experiment(train, validation, vocab, data_id, config, output, torch.device('cpu'))
    report = json.loads((output / 'summary.json').read_text(encoding='utf-8'))
    assert all(row['validation_tokens'] == 9 for row in report['runs'])
print('Các số trên chỉ kiểm đường chạy CPU; không đưa vào bảng kết quả Truyện Kiều.')

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_char_lm*.py', '-v'], cwd=root, check=True)

### Thí nghiệm chính thức — chạy trên Google Colab khi đủ điều kiện
1. Dùng [corpus Trương Vĩnh Ký 1911 đã chuẩn bị](../docs/data/char-corpus.md), nguồn Wikisource `206466` đối chiếu bản scan. Người duyệt của nhóm kiểm hồ sơ, rồi lưu xác nhận quyền huấn luyện và xử lý Colab gắn với SHA-256 manifest. Không dùng `206607` hoặc bản audit.
2. Chọn runtime GPU. Đặt `MANIFEST`, `ARTIFACT_DIR` và `RIGHTS_ATTESTATION` đến đúng hiện vật; đặt `OUTPUT` đến một **thư mục mới trong kho riêng bền vững mà bạn đã cấp quyền** (không chỉ `/content`). Đổi `RUN_KIEU=True`. Không cần GPU cho các ô minh họa trước.
3. Cấu hình đề xuất: 2 decoder, chiều 64, 4 head, FFN 128, batch 16, AdamW với LR 0,001, 5 epoch; T lấy từ manifest (đề xuất 128). Không phải cấu hình đã tối ưu. Seed 41–45 dùng cùng split. Không dùng test. Khi ngắt phiên, mở lại đúng revision mã, hiện vật, cấu hình, môi trường và đường dẫn output; đặt `RESUME=True` để tiếp tục từ epoch đã lưu, không tính lại thành seed mới.
4. Sau đủ 5 seed, lưu `summary.json` và checkpoint ở kho riêng. Báo từng PPL validation, số target và trung bình ± SD mẫu qua **5 lần huấn luyện**; không thay bằng 5 lần sinh từ một checkpoint. Không đưa checkpoint, corpus hay hồ sơ quyền riêng tư vào Git.

In [ ]:
RUN_KIEU = False
RESUME = False
MANIFEST = root / '.cache/char-corpus/manifest-1911.json'
ARTIFACT_DIR = root / '.cache/char-corpus/prepared-1911'
RIGHTS_ATTESTATION = root / '.cache/char-corpus/rights-1911.json'
OUTPUT = None  # Đặt Path tới thư mục đích bền vững trước khi bật RUN_KIEU.
if RUN_KIEU:
    if OUTPUT is None:
        raise ValueError('Cần cấu hình OUTPUT trước khi chạy.')
    cmd = [sys.executable, '-m', 'scripts.train_char_lm', '--manifest', str(MANIFEST),
           '--artifact-dir', str(ARTIFACT_DIR), '--rights-attestation', str(RIGHTS_ATTESTATION),
           '--output', str(OUTPUT), '--device', 'cuda']
    subprocess.run(cmd + (['--resume'] if RESUME else []), cwd=root, check=True)
else:
    print('Chưa chạy trên Truyện Kiều. Không có PPL chính thức từ notebook mặc định.')